In [13]:
import gc
import warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

# Primary GBDT model
import lightgbm as lgb

# Optional high-performance gradient boosters
try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

from sklearn.ensemble import HistGradientBoostingClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

print(f"LightGBM Version: {lgb.__version__}")
print(f"CatBoost Available: {CATBOOST_AVAILABLE}")
print(f"XGBoost Available: {XGBOOST_AVAILABLE}")

LightGBM Version: 4.7.0
CatBoost Available: True
XGBoost Available: True


In [14]:
print("Loading supplementary datasets...")

# 1. Customers: standardize messy region strings and retain latest customer record
df_customers = pd.read_csv('customers.csv')
df_customers['region'] = (
    df_customers['region']
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r'[\s_-]+', '-', regex=True)
)
df_customers['signup_date'] = pd.to_datetime(df_customers['signup_date'], errors='coerce')
df_customers['record_updated_at'] = pd.to_datetime(df_customers['record_updated_at'], errors='coerce')
df_customers = df_customers.sort_values('record_updated_at').drop_duplicates(subset=['customer_id'], keep='last')

# 2. Products: dimensions, density, and category-level relative benchmarks
df_products = pd.read_csv('products.csv')
df_products['launch_date'] = pd.to_datetime(df_products['launch_date'], errors='coerce')
df_products['vol_cm3'] = df_products['length_cm'] * df_products['width_cm'] * df_products['height_cm']
df_products['density'] = df_products['weight_kg'] / (df_products['vol_cm3'] + 1e-4)

cat_benchmarks = df_products.groupby('category').agg(
    cat_mean_price=('list_price', 'mean'),
    cat_mean_weight=('weight_kg', 'mean')
).reset_index()

df_products = df_products.merge(cat_benchmarks, on='category', how='left')
df_products['price_to_cat_mean'] = df_products['list_price'] / (df_products['cat_mean_price'] + 1e-4)
df_products['weight_to_cat_mean'] = df_products['weight_kg'] / (df_products['cat_mean_weight'] + 1e-4)

# 3. Stores: standardize region format
df_stores = pd.read_csv('stores.csv')
df_stores['opened_date'] = pd.to_datetime(df_stores['opened_date'], errors='coerce')
df_stores['region'] = (
    df_stores['region']
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r'[\s_-]+', '-', regex=True)
)

# 4. Calendar: temporal economic context
df_calendar = pd.read_csv('calendar.csv')
df_calendar['week_start'] = pd.to_datetime(df_calendar['week_start'])

print("Supplementary tables loaded and normalized.")

Loading supplementary datasets...
Supplementary tables loaded and normalized.


In [15]:
def transform_transactions(df):
    df['order_ts'] = pd.to_datetime(df['order_ts'])
    df['week_start'] = df['order_ts'].dt.to_period('W-SUN').apply(lambda r: r.start_time)

    # Relational joins
    df = df.merge(df_customers, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(df_products, on='product_id', how='left', suffixes=('', '_prod'))
    df = df.merge(df_stores, on='store_id', how='left', suffixes=('', '_store'))
    df = df.merge(df_calendar, on='week_start', how='left')

    # Macro temporal components (excluding day-of-month to avoid memorizing historical calendar dates)
    df['order_year'] = df['order_ts'].dt.year.astype(np.int16)
    df['order_month'] = df['order_ts'].dt.month.astype(np.int8)
    df['order_dayofweek'] = df['order_ts'].dt.dayofweek.astype(np.int8)
    df['is_weekend'] = df['order_dayofweek'].isin([5, 6]).astype(np.int8)

    # Durations and account maturity (in days)
    df['days_since_signup'] = (df['order_ts'] - df['signup_date']).dt.days.fillna(-1).clip(lower=-1)
    df['days_since_launch'] = (df['order_ts'] - df['launch_date']).dt.days.fillna(-1).clip(lower=-1)
    df['days_since_store_open'] = (df['order_ts'] - df['opened_date']).dt.days.fillna(-1).clip(lower=-1)

    # Pricing, discount, and exposure ratios
    df['line_raw_val'] = df['unit_price'] * df['quantity']
    df['calc_line_total'] = df['line_raw_val'] * (1.0 - df['discount'].fillna(0))
    df['price_ratio_list'] = df['unit_price'] / (df['list_price'] + 1e-4)
    df['discount_diff'] = (df['list_price'] - df['unit_price']).clip(lower=0)
    df['window_price_exposure'] = df['return_window_days'] * np.log1p(df['unit_price'].clip(lower=0))

    # Shipping and delivery friction
    df['freight_share'] = df['freight_cost'] / (df['order_value'] + 1e-4)
    df['freight_per_unit'] = df['freight_cost'] / (df['quantity'] + 1e-4)
    df['freight_per_kg'] = df['freight_cost'] / (df['weight_kg'].fillna(1.0) + 1e-4)
    df['cross_region_shipping'] = (df['region'] != df['region_store']).astype(np.int8)

    # Session intent
    df['has_coupon'] = df['coupon_code'].notnull().astype(np.int8)
    df['clicks_per_item'] = df['session_clicks'].fillna(0) / (df['items_in_order'] + 1e-4)
    df['line_position_ratio'] = df['line_no'] / (df['items_in_order'] + 1e-4)

    # Macroeconomic context
    df['cost_pressure'] = df['fuel_price'] * (df['fuel_surcharge'].fillna(0) + 1.0)
    df['price_to_cpi'] = df['unit_price'] / (df['cpi'] + 1e-4)

    # Fast vectorized basket / bracketing aggregations
    basket_stats = df.groupby('order_id').agg(
        order_line_count=('order_line_id', 'count'),
        order_total_qty=('quantity', 'sum'),
        order_max_price=('unit_price', 'max'),
        order_distinct_prods=('product_id', 'nunique')
    ).reset_index()

    df = df.merge(basket_stats, on='order_id', how='left')
    df['line_qty_share'] = df['quantity'] / (df['order_total_qty'] + 1e-4)
    df['price_rel_to_max'] = df['unit_price'] / (df['order_max_price'] + 1e-4)

    # Drop raw datetimes and high-noise warehouse artifacts
    drop_cols = [
        'order_ts', 'signup_date', 'launch_date', 'opened_date',
        'record_updated_at', 'week_start', 'coupon_code',
        'warehouse_zone', 'picker_shift'
    ]
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
    return df

In [16]:
print("Loading train.csv and test.csv...")
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

print("Applying feature engineering transformations...")
df_train = transform_transactions(df_train)
df_test = transform_transactions(df_test)

# Ignore primary ID columns and the target label
ignore_cols = ['order_line_id', 'order_id', 'customer_id', 'product_id', 'store_id', 'returned']
features = [c for c in df_train.columns if c not in ignore_cols]

# 1. Frequency encoding for high-cardinality entities
for col in ['supplier_code', 'banner_id', 'store_id', 'category', 'promo_channel_code']:
    if col in df_train.columns:
        freq = df_train[col].value_counts(normalize=True).to_dict()
        df_train[f'{col}_freq'] = df_train[col].map(freq).fillna(0).astype(np.float32)
        df_test[f'{col}_freq'] = df_test[col].map(freq).fillna(0).astype(np.float32)
        features.append(f'{col}_freq')

# 2. Identify ALL non-numeric / string columns (regardless of str/object/category dtype)
non_numeric_cols = [c for c in features if not pd.api.types.is_numeric_dtype(df_train[c])]
print(f"Safely encoding {len(non_numeric_cols)} string/categorical columns into numeric IDs:")
print(non_numeric_cols)

for col in non_numeric_cols:
    df_train[col] = df_train[col].astype(str)
    df_test[col] = df_test[col].astype(str)
    cats, uniques = pd.factorize(df_train[col])
    df_train[col] = cats.astype(np.int32)
    mapping = {val: idx for idx, val in enumerate(uniques)}
    df_test[col] = df_test[col].map(mapping).fillna(-1).astype(np.int32)

# Ensure strictly numeric features
features = sorted(list(dict.fromkeys([c for c in features if pd.api.types.is_numeric_dtype(df_train[c])])))
print(f"\nFinal feature set verified: {len(features)} numeric columns.")
gc.collect()

Loading train.csv and test.csv...
Applying feature engineering transformations...
Safely encoding 18 string/categorical columns into numeric IDs:
['tz_hint', 'channel', 'payment_method', 'ship_mode', 'promo_channel_code', 'device_type', 'region', 'customer_type', 'age_band', 'preferred_contact', 'loyalty_tier', 'category', 'packaging_type', 'supplier_code', 'color', 'channel_store', 'region_store', 'format']

Final feature set verified: 83 numeric columns.


0

In [17]:
# Validation split: Late 2024 holdout (Months 21-24) to simulate future test period (Months 25-30)
is_val = (df_train['order_year'] == 2024) & (df_train['order_month'] >= 9)

# Explicitly cast to float32 to guarantee complete compatibility across all GBDT libraries
X_tr = df_train.loc[~is_val, features].astype(np.float32)
y_tr = df_train.loc[~is_val, 'returned'].astype(np.int32)

X_val = df_train.loc[is_val, features].astype(np.float32)
y_val = df_train.loc[is_val, 'returned'].astype(np.int32)

X_test = df_test[features].astype(np.float32)

print(f"Train records: {len(X_tr):,} | Holdout validation records: {len(X_val):,}")
print(f"Historical return rate: {y_tr.mean():.4f}")

# Containers for model comparison and ensembling
val_predictions = {}
test_predictions = {}

Train records: 1,165,899 | Holdout validation records: 234,106
Historical return rate: 0.1804


In [18]:
print("--- Training Model 1: Regularized LightGBM ---")

train_lgb = lgb.Dataset(X_tr, label=y_tr)
val_lgb = lgb.Dataset(X_val, label=y_val, reference=train_lgb)

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 31,              # Constrained to prevent memorizing small subgroups
    'max_depth': 6,                # Shallow depth avoids fitting high-order noise
    'min_child_samples': 500,      # Requires substantial mass per leaf across 1M+ rows
    'feature_fraction': 0.70,      # Column subsampling per tree
    'bagging_fraction': 0.75,      # Row subsampling
    'bagging_freq': 1,
    'reg_alpha': 3.0,              # L1 regularization
    'reg_lambda': 15.0,            # L2 regularization
    'n_jobs': -1,
    'random_state': 42,
    'verbose': -1
}

model_lgb = lgb.train(
    lgb_params,
    train_lgb,
    num_boost_round=2500,
    valid_sets=[train_lgb, val_lgb],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=60),
        lgb.log_evaluation(period=100)
    ]
)

val_predictions['LightGBM'] = model_lgb.predict(X_val, num_iteration=model_lgb.best_iteration)
test_predictions['LightGBM'] = model_lgb.predict(X_test, num_iteration=model_lgb.best_iteration)

auc_lgb = roc_auc_score(y_val, val_predictions['LightGBM'])
print(f"\n--> LightGBM Validation ROC-AUC: {auc_lgb:.5f} <--")

--- Training Model 1: Regularized LightGBM ---
Training until validation scores don't improve for 60 rounds
[100]	train's auc: 0.727988	val's auc: 0.627943
[200]	train's auc: 0.733944	val's auc: 0.644888
[300]	train's auc: 0.737202	val's auc: 0.652689
[400]	train's auc: 0.739815	val's auc: 0.658237
[500]	train's auc: 0.741798	val's auc: 0.661569
[600]	train's auc: 0.743684	val's auc: 0.664846
[700]	train's auc: 0.745339	val's auc: 0.667726
[800]	train's auc: 0.74699	val's auc: 0.670091
[900]	train's auc: 0.748431	val's auc: 0.671548
[1000]	train's auc: 0.749862	val's auc: 0.672986
[1100]	train's auc: 0.751138	val's auc: 0.6742
[1200]	train's auc: 0.752385	val's auc: 0.675154
[1300]	train's auc: 0.753555	val's auc: 0.675698
[1400]	train's auc: 0.754738	val's auc: 0.676554
[1500]	train's auc: 0.755917	val's auc: 0.677096
[1600]	train's auc: 0.757059	val's auc: 0.677552
[1700]	train's auc: 0.758211	val's auc: 0.678082
[1800]	train's auc: 0.759343	val's auc: 0.678635
[1900]	train's auc: 0.

In [19]:
if CATBOOST_AVAILABLE:
    print("--- Training Model 2: Regularized CatBoost ---")
    model_cb = CatBoostClassifier(
        iterations=1500,
        learning_rate=0.04,
        depth=6,
        l2_leaf_reg=12.0,
        eval_metric='AUC',
        random_seed=42,
        task_type='CPU',
        early_stopping_rounds=50,
        verbose=100
    )
    model_cb.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
    val_predictions['CatBoost'] = model_cb.predict_proba(X_val)[:, 1]
    test_predictions['CatBoost'] = model_cb.predict_proba(X_test)[:, 1]
    auc_cb = roc_auc_score(y_val, val_predictions['CatBoost'])
    print(f"\n--> CatBoost Validation ROC-AUC: {auc_cb:.5f} <--")
else:
    print("--- Training Model 2: Sklearn HistGradientBoosting ---")
    model_hgb = HistGradientBoostingClassifier(
        loss='log_loss',
        learning_rate=0.04,
        max_iter=600,
        max_leaf_nodes=31,
        min_samples_leaf=150,
        l2_regularization=10.0,
        early_stopping=True,
        random_state=42
    )
    model_hgb.fit(X_tr, y_tr)
    val_predictions['HistGBM'] = model_hgb.predict_proba(X_val)[:, 1]
    test_predictions['HistGBM'] = model_hgb.predict_proba(X_test)[:, 1]
    auc_hgb = roc_auc_score(y_val, val_predictions['HistGBM'])
    print(f"\n--> HistGradientBoosting Validation ROC-AUC: {auc_hgb:.5f} <--")

--- Training Model 2: Regularized CatBoost ---
0:	test: 0.5787420	best: 0.5787420 (0)	total: 466ms	remaining: 11m 38s
100:	test: 0.6298308	best: 0.6298308 (100)	total: 21s	remaining: 4m 51s
200:	test: 0.6440345	best: 0.6440345 (200)	total: 42.4s	remaining: 4m 34s
300:	test: 0.6505019	best: 0.6505019 (300)	total: 1m 1s	remaining: 4m 6s
400:	test: 0.6578359	best: 0.6578359 (400)	total: 1m 24s	remaining: 3m 50s
500:	test: 0.6634583	best: 0.6634583 (500)	total: 1m 44s	remaining: 3m 29s
600:	test: 0.6664736	best: 0.6664736 (600)	total: 2m 3s	remaining: 3m 5s
700:	test: 0.6695173	best: 0.6695173 (700)	total: 2m 22s	remaining: 2m 41s
800:	test: 0.6710513	best: 0.6710513 (800)	total: 2m 41s	remaining: 2m 20s
900:	test: 0.6729600	best: 0.6729600 (900)	total: 2m 59s	remaining: 1m 59s
1000:	test: 0.6738647	best: 0.6738647 (1000)	total: 3m 18s	remaining: 1m 38s
1100:	test: 0.6779262	best: 0.6779262 (1100)	total: 3m 36s	remaining: 1m 18s
1200:	test: 0.6800833	best: 0.6800871 (1197)	total: 3m 55s	re

In [20]:
if XGBOOST_AVAILABLE:
    print("--- Training Model 3: XGBoost (Hist Mode) ---")
    model_xgb = xgb.XGBClassifier(
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=30,
        subsample=0.75,
        colsample_bytree=0.70,
        reg_alpha=3.0,
        reg_lambda=15.0,
        tree_method='hist',
        eval_metric='auc',
        early_stopping_rounds=60,
        random_state=42,
        n_jobs=-1
    )
    model_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)
    val_predictions['XGBoost'] = model_xgb.predict_proba(X_val)[:, 1]
    test_predictions['XGBoost'] = model_xgb.predict_proba(X_test)[:, 1]
    auc_xgb = roc_auc_score(y_val, val_predictions['XGBoost'])
    print(f"\n--> XGBoost Validation ROC-AUC: {auc_xgb:.5f} <--")
else:
    print("XGBoost is not installed. Skipping Model 3.")

--- Training Model 3: XGBoost (Hist Mode) ---
[0]	validation_0-auc:0.60759
[100]	validation_0-auc:0.63450
[200]	validation_0-auc:0.64892
[300]	validation_0-auc:0.65612
[400]	validation_0-auc:0.66086
[500]	validation_0-auc:0.66408
[600]	validation_0-auc:0.66731
[700]	validation_0-auc:0.66957
[800]	validation_0-auc:0.67152
[900]	validation_0-auc:0.67284
[1000]	validation_0-auc:0.67434
[1100]	validation_0-auc:0.67516
[1200]	validation_0-auc:0.67598
[1300]	validation_0-auc:0.67638
[1400]	validation_0-auc:0.67706
[1499]	validation_0-auc:0.67752

--> XGBoost Validation ROC-AUC: 0.67753 <--


In [21]:
print("\n" + "="*50)
print("FINAL VALIDATION SCOREBOARD")
print("="*50)

model_scores = {}
for name, preds in val_predictions.items():
    score = roc_auc_score(y_val, preds)
    model_scores[name] = score
    print(f"{name:<20} : {score:.5f}")

# Rank-Average Ensemble across all trained models
if len(val_predictions) > 1:
    ens_val = np.zeros(len(y_val))
    ens_test = np.zeros(len(df_test))
    
    for name in val_predictions.keys():
        ens_val += rankdata(val_predictions[name]) / len(val_predictions[name])
        ens_test += rankdata(test_predictions[name]) / len(test_predictions[name])
        
    ens_val /= len(val_predictions)
    ens_test /= len(test_predictions)
    
    ens_score = roc_auc_score(y_val, ens_val)
    print(f"{'RANK ENSEMBLE':<20} : {ens_score:.5f}")
    
    best_single_name = max(model_scores, key=model_scores.get)
    best_single_score = model_scores[best_single_name]
    
    if ens_score >= best_single_score:
        print(f"\n--> Selecting Rank Ensemble (ROC-AUC: {ens_score:.5f}) for final submission")
        final_test_preds = ens_test
    else:
        print(f"\n--> Selecting {best_single_name} (ROC-AUC: {best_single_score:.5f}) for final submission")
        final_test_preds = test_predictions[best_single_name]
else:
    best_name = list(val_predictions.keys())[0]
    final_test_preds = test_predictions[best_name]

# Save submission matching sample_submission.csv format
submission = pd.DataFrame({
    'order_line_id': df_test['order_line_id'],
    'returned': final_test_preds
})

submission.to_csv('submission.csv', index=False)
print("\nSubmission file saved to submission.csv successfully!")
print(f"Shape: {submission.shape}")
print(submission.head(10))


FINAL VALIDATION SCOREBOARD
LightGBM             : 0.68098
CatBoost             : 0.68317
XGBoost              : 0.67753
RANK ENSEMBLE        : 0.68184

--> Selecting CatBoost (ROC-AUC: 0.68317) for final submission

Submission file saved to submission.csv successfully!
Shape: (479995, 2)
     order_line_id  returned
0  L85d608999fa1ee  0.117746
1  L86272cb9543018  0.268229
2  L66b5669bdace5b  0.098846
3  L2c5886124612de  0.165109
4  L56fe7aa5ac5dd8  0.123903
5  L40cee0cd374598  0.121890
6  Lf3f152acf5df47  0.279427
7  L67e552e178dbb2  0.123315
8  L8070dab1c78a1e  0.107783
9  Ldf99a128238815  0.120410
